# Networking & SSH

## What's covered

- **TCP/IP refresher** — IP addresses, ports, hostnames, IPv4 vs IPv6, just enough to be productive
- **Network interfaces** — what they are, how Linux names them (`lo`, `eth0`, `enp0s3`), the modern **`ip`** command
- **Static vs DHCP** — the two ways an interface gets an address
- **NetworkManager / `nmcli`** — the modern config tool on most desktops and many servers
- **Routing** — default gateway, the routing table, `ip route`
- **DNS resolution** — `/etc/resolv.conf`, `systemd-resolved`, `/etc/hosts`, `dig` / `nslookup` / `host`
- **SSH** — the secure shell: client, keys, agent, config (`~/.ssh/config`), server (`/etc/ssh/sshd_config`)
- **File transfer over SSH** — `scp` and **`rsync`** (the right tool for bigger jobs)
- **Firewalls** — `iptables` / `nftables` (the kernel layer), **`ufw`** (Debian/Ubuntu friendly UI), **`firewalld`** (RHEL/Fedora friendly UI)
- **Troubleshooting** — `ping`, `traceroute`, `ss`, `dig`, `tcpdump`, and the OSI-layered flow for diagnosing "the network is broken"

This notebook assumes notebooks 01-07. Networking is one of the larger LFCS topics. By the end you'll be able to inspect and configure interfaces, set up SSH access with keys, transfer files, configure a basic firewall, and diagnose common network problems.

## TCP/IP refresher

A Linux box on a network has three layers you need to understand to read most output:

- **IP address** — a number that identifies a machine on a network. IPv4: four numbers 0-255 separated by dots (e.g. `192.168.1.42`). IPv6: eight groups of hex separated by colons (e.g. `2001:db8::1`). Every interface gets at least one.
- **Port** — a number 0-65535 that identifies a specific service on a machine. Web servers usually listen on **80** (HTTP) and **443** (HTTPS); SSH on **22**; PostgreSQL on **5432**. The combination of IP + port uniquely identifies a network endpoint.
- **Protocol** — how the two ends talk. **TCP** is connection-oriented, reliable, ordered (used by HTTP, SSH, SMTP, most services). **UDP** is connectionless, fast, lossy (used by DNS, DHCP, video streaming, game traffic).

### IP addressing in brief

An IPv4 address has 32 bits. A **network** is a range of addresses; a single machine has one address. You write a network in **CIDR** notation (Classless Inter-Domain Routing): `address/prefix-length`.

| CIDR | Range | Used for |
|---|---|---|
| `192.168.1.0/24` | `192.168.1.0` – `192.168.1.255` (256 addresses) | typical home LAN |
| `10.0.0.0/8` | `10.0.0.0` – `10.255.255.255` (~16.7M) | corporate / cloud networks |
| `172.16.0.0/12` | `172.16.0.0` – `172.31.255.255` (~1M) | corporate networks |
| `127.0.0.0/8` | `127.0.0.1` is "localhost" — this machine | loopback |

The three ranges above (`10.x`, `172.16-31.x`, `192.168.x`) are **private** — they're never routed on the public internet, so home and corporate networks all use them. Your laptop on Wi-Fi probably has a `192.168.x.x` address.

A **gateway** (or "default gateway") is the address packets get sent to when their destination isn't on your local network. Usually your router. Without a default gateway, you can talk to local machines but not the wider internet.

For LFCS, you don't need to be a network engineer. Understand the address/port/protocol model, that `/24` means a 256-address LAN, and that you need a gateway for traffic beyond the LAN.

## Network interfaces

A **network interface** is the kernel's abstraction for a network connection — a physical Ethernet port, a Wi-Fi card, a USB ethernet adapter, a virtual interface for a VPN or container. Each one has a name and (usually) at least one IP address.

Common interface names you'll see:

| Name | What it is |
|---|---|
| **`lo`** | loopback — the virtual interface for "talking to yourself" (`127.0.0.1`). Always present. |
| **`eth0`, `eth1`** | legacy ethernet naming — still common in VMs, containers, embedded systems |
| **`enp0s3`, `enp3s0`** | modern "predictable" ethernet naming (since systemd) — based on bus/slot |
| **`wlan0`, `wlp3s0`** | wireless |
| **`docker0`** | the bridge created by Docker for container networking |
| **`virbr0`** | KVM/libvirt bridge |
| **`tun0`, `tap0`** | VPN tunnels (OpenVPN, WireGuard, etc.) |
| **`br0`** | a user-defined bridge |

### The `ip` command — the modern unified tool

In older tutorials you'll see commands like `ifconfig`, `route`, `arp`, `netstat`. These are **deprecated**. They've been replaced by **`ip`** (interfaces and routes) and **`ss`** (sockets) — both from the `iproute2` package. Use these in new work.

```bash
ip addr                          # all interfaces and their addresses (short: 'ip a')
ip link                          # interface state only, no addresses
ip route                         # routing table
ip -s link                       # statistics (bytes, packets, errors)
ip -br addr                      # brief, one line per interface — easier to read
```

`ip` operates on objects: `link`, `addr`, `route`, `neigh`, `rule`. Within each object you have actions: `show`, `add`, `del`, `set`. For example:

```bash
sudo ip link set eth0 up                  # bring interface up
sudo ip link set eth0 down                # bring it down
sudo ip addr add 192.168.1.50/24 dev eth0 # add an address (temporary)
sudo ip addr del 192.168.1.50/24 dev eth0 # remove it
```

These changes are **temporary** — they vanish at next reboot. For persistent config, use NetworkManager or your distro's config files (next sections).

In [ ]:
!ip -br addr 2>/dev/null || ifconfig | head -10
!ip link 2>/dev/null | head -20

## Getting an IP — static vs DHCP

An interface needs an address before it can communicate. Two ways:

**DHCP** (Dynamic Host Configuration Protocol) — the interface broadcasts "anyone got a config for me?" and a DHCP server (your router, or a corporate server) replies with an address, subnet mask, gateway, and DNS servers. The client periodically renews the lease (typically every few hours). **Most laptops, desktops, VMs, and cloud instances use DHCP**.

**Static** — you configure the address by hand. The machine never asks anyone; it just has that address. Used for servers that need a predictable address (a DNS server, a router, a web server), printers, network equipment.

You decide static vs DHCP per-interface in your network configuration. The tool depends on the distro:

| Distro family | Config tool |
|---|---|
| Ubuntu Desktop, Fedora, RHEL, most modern installs | **NetworkManager** (CLI: `nmcli`) |
| Ubuntu Server | **`netplan`** (YAML config → applied via NetworkManager or systemd-networkd) |
| Minimal / cloud images | **`systemd-networkd`** with `/etc/systemd/network/*.network` files |
| Older RHEL/CentOS | scripts in `/etc/sysconfig/network-scripts/` |
| Debian classic | `/etc/network/interfaces` |

For LFCS, expect **NetworkManager + `nmcli`** or **`netplan`** on Ubuntu Server. Most distros today use NetworkManager.

## `nmcli` — NetworkManager from the command line

NetworkManager (the daemon, `NetworkManager`) is the modern Linux network config service. **`nmcli`** is its command-line interface — the one you'll use on a server without a GUI.

`nmcli` works on two concepts:

- **Devices** — physical/virtual interfaces (what `ip link` shows).
- **Connections** — configuration profiles that can be activated on a device. You can have multiple connections per device (home Wi-Fi, office Wi-Fi, etc.); only one is active at a time.

### Inspection

```bash
nmcli                                         # full status (devices + connections)
nmcli device status                           # one line per device
nmcli connection show                         # all connection profiles
nmcli connection show "Wired connection 1"    # details of one profile
```

### Creating a static IP connection

```bash
# Add a new connection profile on eth0, with a static IP
sudo nmcli connection add type ethernet \
    con-name "office-static" ifname eth0 \
    ipv4.method manual \
    ipv4.addresses 192.168.1.50/24 \
    ipv4.gateway 192.168.1.1 \
    ipv4.dns "8.8.8.8 1.1.1.1"

# Activate it
sudo nmcli connection up office-static
```

To set DHCP instead, use `ipv4.method auto` and skip the address/gateway/dns fields.

### Modifying an existing connection

```bash
sudo nmcli connection modify office-static ipv4.dns "9.9.9.9"
sudo nmcli connection up office-static       # reload for changes to take effect
```

### Deleting

```bash
sudo nmcli connection down office-static
sudo nmcli connection delete office-static
```

### Wi-Fi quick reference

```bash
nmcli device wifi list                        # available networks
sudo nmcli device wifi connect SSID password 'PASS'
nmcli radio wifi off                          # turn Wi-Fi off
nmcli radio wifi on
```

For LFCS, know **`nmcli connection add`** with `ipv4.method manual` and the four key fields (addresses, gateway, dns, method), and know how to bring connections up/down.

In [ ]:
!nmcli device status 2>/dev/null
!nmcli connection show 2>/dev/null

## Routing — how packets find their destination

When your machine wants to send a packet to `93.184.216.34` (example.com), the kernel consults its **routing table** to decide *which interface* to send it through and *which next hop* to send it to.

A routing table is a list of "if the destination matches this network, use this gateway via this interface." The kernel finds the **most specific** matching entry.

To see your routes:

```bash
ip route                         # the modern way
ip -6 route                      # IPv6 routes
```

A typical output:

```
default via 192.168.1.1 dev eth0 proto dhcp metric 100
192.168.1.0/24 dev eth0 proto kernel scope link src 192.168.1.42
```

Read it line by line:

- **`default via 192.168.1.1 dev eth0`** — for anything that doesn't match another rule, send it to `192.168.1.1` via interface `eth0`. This is the **default gateway** (sometimes called "default route"). The router on a home LAN; in the cloud, a virtual gateway provided by the VPC.
- **`192.168.1.0/24 dev eth0 ... src 192.168.1.42`** — for the local network `192.168.1.0/24`, no gateway needed (it's directly attached); use interface `eth0`, source address `192.168.1.42`.

Most boxes have just these two rules: "local network is reachable directly" + "everything else goes via the gateway." More complex setups (VPNs, multi-homed servers) add more rules.

### Manipulating routes

```bash
sudo ip route add 10.0.0.0/8 via 192.168.1.254   # route to 10.0.0.0/8 via this gateway
sudo ip route del 10.0.0.0/8                     # remove
sudo ip route change default via 192.168.1.2     # change the default gateway
```

These are temporary; for persistence, configure them through NetworkManager (`ipv4.routes`) or your distro's network config.

### Finding which interface will be used for a destination

```bash
ip route get 8.8.8.8                             # which interface + gateway for this destination
```

Use this when debugging: "is my traffic actually going where I think it is?"

In [ ]:
!ip route 2>/dev/null
!ip route get 1.1.1.1 2>/dev/null

## DNS — turning names into addresses

You don't connect to `93.184.216.34`. You connect to `example.com`. **DNS (Domain Name System)** is what turns the name into the address.

The lookup typically goes:

1. **`/etc/hosts`** — a local file with hardcoded name-to-address mappings. Checked first by default.
2. **The system DNS resolver** — usually `systemd-resolved` or a stub specified in `/etc/resolv.conf`.
3. **The configured DNS servers** — your ISP's, your corporate DNS, or public ones (`8.8.8.8`, `1.1.1.1`, `9.9.9.9`).
4. **The wider DNS** — root servers, TLD servers, authoritative nameservers — handled by your configured resolver.

### `/etc/hosts`

A simple text file mapping IPs to names. Useful for blocking sites, overriding DNS for development, giving local machines memorable names:

```
127.0.0.1       localhost
::1             localhost ip6-localhost
192.168.1.10    home-server home-server.local
192.168.1.11    printer
```

The format is one mapping per line: address, then one or more names. Beats every other source by default.

### `/etc/resolv.conf` and `systemd-resolved`

`/etc/resolv.conf` historically held the list of DNS servers. Two lines you'll see:

```
nameserver 8.8.8.8
nameserver 1.1.1.1
search example.com
```

`nameserver` lines list resolvers in priority order. `search` lists domains to try as suffixes for short names (so `ping web` tries `web.example.com` if just `web` doesn't resolve).

On modern systemd distros, **`/etc/resolv.conf` is a symlink to a file managed by `systemd-resolved`**. Don't edit it directly — it'll get overwritten. Instead:

```bash
resolvectl status                             # show current DNS config
resolvectl query example.com                  # do a lookup through systemd-resolved
sudo resolvectl dns eth0 8.8.8.8 1.1.1.1      # set DNS for an interface temporarily
```

For permanent changes, set DNS through NetworkManager (`ipv4.dns` as shown above) or edit `/etc/systemd/resolved.conf`.

### `dig`, `nslookup`, `host` — manual DNS queries

When you want to *test* DNS without trusting the local cache:

```bash
dig example.com                               # full DNS query with timings (best for debugging)
dig example.com +short                        # just the answer
dig @8.8.8.8 example.com                      # query a specific server (bypass /etc/resolv.conf)
dig example.com MX                            # query a specific record type (A, MX, NS, TXT, etc.)
dig -x 93.184.216.34                          # reverse lookup (address → name)

nslookup example.com                          # simpler output, older tool
host example.com                              # one-line output, easy to parse
```

**`dig` is the LFCS-relevant one.** Use `dig @8.8.8.8 example.com` when you suspect your local DNS is broken — if Google's DNS works, the problem is your resolver, not the wider internet.

Record types worth knowing:

- **A** — IPv4 address
- **AAAA** — IPv6 address
- **CNAME** — alias to another name
- **MX** — mail server for the domain
- **NS** — nameservers for the domain
- **TXT** — arbitrary text (used for SPF, DKIM, domain verification)
- **PTR** — reverse lookup (address → name)

In [ ]:
!cat /etc/resolv.conf 2>/dev/null
!resolvectl status 2>/dev/null | head -20 || cat /etc/resolv.conf
!dig +short example.com 2>/dev/null
!dig example.com MX +short 2>/dev/null

## SSH — the secure shell

**SSH (Secure SHell)** is how you log in to remote Linux machines and run commands as if you were at the console. It encrypts everything, authenticates both sides, and is the universal admin protocol — used the same way on every Linux box, every cloud provider, every container.

The two ends:

- **SSH client** — the `ssh` command you run on your machine.
- **SSH server** — the `sshd` daemon running on the remote machine, listening on TCP port **22** by default.

### Basic usage

```bash
ssh user@host                                 # log in as 'user' on 'host'
ssh user@host -p 2222                         # custom port
ssh -i ~/.ssh/special_key user@host           # use a specific key
ssh user@host "hostname && uptime"            # run one command and exit
ssh -t user@host "sudo something"             # allocate a pseudo-terminal (needed for sudo prompt)
```

Without a port flag, SSH connects to port 22. Many security-conscious sites move SSH to a non-standard port to reduce noise from scanners — but the right answer is good keys, not port obscurity.

### Host key verification

The first time you connect to a host, SSH shows you the host's **fingerprint** and asks if it's correct:

```
The authenticity of host 'example.com (93.184.216.34)' can't be established.
ED25519 key fingerprint is SHA256:abc...xyz.
Are you sure you want to continue connecting (yes/no/[fingerprint])?
```

This protects against man-in-the-middle attacks. On first connection, **verify the fingerprint out-of-band** (ask the admin, check the cloud provider's console). Once accepted, it's stored in `~/.ssh/known_hosts` and checked silently on every subsequent connection.

If you ever get this warning:

```
@    WARNING: REMOTE HOST IDENTIFICATION HAS CHANGED!    @
```

— it means the host's key changed since your last connection. Usually because the server was reinstalled or moved IP. **Don't blindly accept it.** Either verify the new key out-of-band, or remove the old entry with `ssh-keygen -R hostname` after confirming the change is legitimate.

## SSH keys — the right way to authenticate

Password authentication works but has two problems: passwords are guessable, and you have to type them every time. **SSH keys** solve both. A key is a cryptographic pair: a **private key** (stays on your machine, never shared) and a **public key** (copied to every server you want to access).

When you connect, the server gives you a challenge that only the holder of the matching private key can answer. No password is sent over the wire; no password is needed by the user.

### Generating a key

The modern recommended algorithm is **Ed25519** — fast, secure, small keys:

```bash
ssh-keygen -t ed25519 -C "your_email@example.com"
```

You'll be prompted for:
- **File location** — default `~/.ssh/id_ed25519` (private) and `~/.ssh/id_ed25519.pub` (public). Press Enter to accept.
- **Passphrase** — encrypts the private key on disk. **Always use one** on a real machine. The agent (next section) means you only type it once per session.

After this, you have two files:

```
~/.ssh/id_ed25519           # PRIVATE — never share, must be mode 600
~/.ssh/id_ed25519.pub       # public — fine to share, copy to servers
```

If you're stuck supporting older systems that don't speak Ed25519, fall back to **RSA** with at least 3072 bits: `ssh-keygen -t rsa -b 3072`.

### Copying the key to a server — `ssh-copy-id`

The easiest way to install your public key on a server:

```bash
ssh-copy-id user@host
```

This copies your public key into `~/.ssh/authorized_keys` on the server (creating it if needed, with the right permissions). After that, `ssh user@host` should log you in without a password.

If `ssh-copy-id` isn't available (rare), do it manually:

```bash
cat ~/.ssh/id_ed25519.pub | ssh user@host 'mkdir -p ~/.ssh && cat >> ~/.ssh/authorized_keys && chmod 600 ~/.ssh/authorized_keys'
```

### Critical permissions

SSH refuses to use keys if permissions are wrong (it's strict on purpose):

```
~/.ssh                       drwx------ (700)
~/.ssh/authorized_keys       -rw------- (600)
~/.ssh/id_ed25519            -rw------- (600)
~/.ssh/id_ed25519.pub        -rw-r--r-- (644)  -- can be world-readable
```

If you get "permissions are too open" errors, fix with:

```bash
chmod 700 ~/.ssh
chmod 600 ~/.ssh/authorized_keys ~/.ssh/id_ed25519
chmod 644 ~/.ssh/id_ed25519.pub
```

In [ ]:
!ls -ld ~/.ssh 2>/dev/null
!ls -la ~/.ssh 2>/dev/null

## SSH config — `~/.ssh/config`

Typing `ssh -i ~/.ssh/special_key -p 2222 user@really.long.hostname.example.com` gets old fast. The SSH config file lets you save aliases.

**`~/.ssh/config`** lets you define per-host settings:

```
Host work
    HostName 10.0.0.42
    User alice
    Port 22
    IdentityFile ~/.ssh/work_ed25519

Host bastion
    HostName bastion.example.com
    User admin
    IdentityFile ~/.ssh/bastion_key

Host internal-*
    User alice
    ProxyJump bastion

Host *
    ServerAliveInterval 60
    ServerAliveCountMax 3
```

Now:

```bash
ssh work                                  # connects as alice to 10.0.0.42 with the right key
ssh internal-db1                          # routes through 'bastion' automatically (ProxyJump)
```

The rules:

- Each `Host` block applies to matching hostnames.
- **First match wins** for each option — order matters.
- **`Host *`** at the end sets defaults for everything.
- Patterns can use glob-style wildcards (`internal-*`, `*.example.com`).

Useful keys:

| Key | Purpose |
|---|---|
| **`HostName`** | actual hostname or IP (the alias can be anything) |
| **`User`** | username |
| **`Port`** | custom port |
| **`IdentityFile`** | which private key to use |
| **`ProxyJump`** | "jump through this host first" — for bastions |
| **`ForwardAgent yes`** | enable agent forwarding (next section) |
| **`ServerAliveInterval`** | seconds between keepalives (keeps idle connections alive through NAT) |
| **`StrictHostKeyChecking`** | `yes` / `no` / `accept-new` — security trade-off |

For LFCS, **know the basic `Host` / `HostName` / `User` / `IdentityFile` pattern**.

## `ssh-agent` and agent forwarding

A passphrase-protected key is more secure but inconvenient — you'd type the passphrase every connection. **`ssh-agent`** holds your decrypted keys in memory so you only enter the passphrase once per session.

### Starting and using the agent

On most desktop Linux systems, the agent is started automatically when you log in. To check:

```bash
ssh-add -l                                # list keys currently held by the agent
```

To add a key:

```bash
ssh-add                                   # add ~/.ssh/id_ed25519 (default location)
ssh-add ~/.ssh/work_ed25519               # add a specific key
ssh-add -D                                # remove all keys from the agent
ssh-add -t 3600 ~/.ssh/work_ed25519       # add with a 1-hour expiration
```

If the agent isn't running, start it:

```bash
eval "$(ssh-agent -s)"
ssh-add
```

### Agent forwarding

When you SSH into a server and from there need to SSH onward to another server, you have three options:

1. **Copy your private key to the first server** — bad. The intermediate server is now another place where your key can be stolen.
2. **Generate a separate key on the intermediate** — okay, but more keys to manage.
3. **Agent forwarding** — let SSH on the first server use your local agent for the next hop. The key never leaves your machine; the first server just gets to *ask* your agent to sign challenges.

Enable agent forwarding with `-A`:

```bash
ssh -A user@bastion
# then from bastion:
ssh internal-db1                          # uses your local agent, not a key on bastion
```

Or in `~/.ssh/config`:

```
Host bastion
    HostName bastion.example.com
    ForwardAgent yes
```

**Security note**: agent forwarding gives the intermediate host the ability to use your key while you're connected. **Don't enable it for untrusted servers.** `ProxyJump` is usually a safer alternative — it tunnels the connection through the bastion without giving the bastion access to your agent.

```
Host internal-db1
    HostName 10.0.0.42
    User alice
    ProxyJump bastion
```

For LFCS, know **`ssh-add`**, **`ssh-add -l`**, and the existence of agent forwarding vs `ProxyJump`.

## The SSH server — `sshd` and `/etc/ssh/sshd_config`

On the server side, the `sshd` daemon listens on port 22 and handles incoming connections. Its config lives in **`/etc/ssh/sshd_config`** (and may be overridden by drop-ins in `/etc/ssh/sshd_config.d/`).

The settings you'll actually touch:

```
Port 22                              # default SSH port; change to non-22 for noise reduction (optional)
PermitRootLogin prohibit-password    # root can log in only with a key, not a password (sensible default)
PermitRootLogin no                   # ...or disable root SSH entirely (more secure; use sudo from a regular account)
PasswordAuthentication no            # require keys; reject password attempts
PubkeyAuthentication yes             # allow keys (default; keep yes)
AllowUsers alice bob ganesh          # whitelist specific users
AllowGroups ssh-users                # ...or use a group
MaxAuthTries 3                       # disconnect after N failed attempts (default 6)
ClientAliveInterval 300              # send keepalive every 5 min to detect dead clients
ClientAliveCountMax 2                # after this many missed responses, disconnect
LogLevel VERBOSE                     # log key fingerprints used — useful for audit
```

After editing, **always test the config and reload before disconnecting**:

```bash
sudo sshd -t                         # syntax check — silent if OK, errors if not
sudo systemctl reload sshd           # apply changes (existing sessions stay alive)
```

If you reload with a broken config, your current SSH session is fine but **new connections will fail**. The drill: keep your current SSH session open, open a *second* SSH session, verify it works, *then* you can disconnect the first.

### Hardening checklist (for production servers)

1. **Disable password auth** — `PasswordAuthentication no`. Keys only.
2. **Disable root login (or restrict to key-only)** — `PermitRootLogin no` or `prohibit-password`.
3. **Use a firewall** to limit who can reach port 22 (`ufw`/`firewalld`, next section).
4. **Install fail2ban** or similar — bans IPs that fail repeatedly.
5. **Monitor `/var/log/auth.log`** (Debian/Ubuntu) or `/var/log/secure` (RHEL) for unusual login activity.

For LFCS, know how to **edit `sshd_config`, validate with `sshd -t`, reload with `systemctl reload sshd`, and the key settings above**.

## File transfer — `scp` and `rsync` over SSH

Two tools, both run over SSH (so they inherit SSH's authentication, encryption, key setup).

### `scp` — secure copy

Like `cp` but works across machines. Syntax mirrors `cp`:

```bash
scp file.txt user@host:/remote/path/                   # local to remote
scp user@host:/remote/file.txt ./                      # remote to local
scp -r dir/ user@host:/remote/path/                    # recursive (directories)
scp -P 2222 file.txt user@host:/path/                  # custom port (NOTE: capital P, unlike ssh's lowercase -p)
scp user@host1:/file.txt user@host2:/path/             # remote to remote
```

`scp` is simple and ubiquitous. For anything more than a few files, **use `rsync` instead**.

### `rsync` — the right tool for serious work

`rsync` is what you reach for when you want:

- **Incremental sync** — only transfer files that changed, only the changed bytes.
- **Resumable** — interrupted transfers can resume where they left off.
- **Two-way mirroring** — keep a directory tree identical between machines.
- **Backups** — much faster than scp for repeated transfers.

Basic syntax:

```bash
rsync -av source/ user@host:/destination/              # archive mode, verbose
rsync -av user@host:/source/ ./local/                  # pull from remote
```

The flags you'll use:

| Flag | Effect |
|---|---|
| **`-a`** | "archive mode" — `-rlptgoD` — recursive + preserve permissions/timestamps/owners/groups/devices |
| **`-v`** | verbose |
| **`-z`** | compress during transfer (saves bandwidth at the cost of CPU) |
| **`-h`** | human-readable sizes |
| **`-P`** | progress + partial — show progress, keep partial files for resume |
| **`--delete`** | also delete files on the destination that aren't on the source (true mirror) |
| **`-n`** or **`--dry-run`** | show what would be transferred, don't actually do it |
| **`--exclude='*.tmp'`** | skip files matching a pattern |
| **`-e ssh`** | force SSH transport (default; mention explicitly if using a custom port: `-e 'ssh -p 2222'`) |

A practical "backup my data to a remote server" command:

```bash
rsync -avzh --delete --exclude='node_modules' \
    ~/work/ user@backup-server:/backups/work/
```

### The trailing slash trick

**This is the gotcha**:

- `rsync -a src dst/` — copies the directory `src` INTO `dst/`, creating `dst/src/`.
- `rsync -a src/ dst/` — copies the CONTENTS of `src` into `dst/`, no `src` subdirectory.

The trailing `/` on the source means "the contents of this directory." Without it, you copy the directory itself. **Many failed backups come from this**. Always think about which you want.

For LFCS, know **`scp src dst`**, **`rsync -av source/ dest/`**, the `-a` flag, **`--delete`** for mirroring, and the trailing-slash rule.

## Firewalls — controlling what can reach you

A **firewall** decides which network packets the kernel accepts and which it drops. Linux has the firewall built into the kernel; userspace tools tell it what rules to apply.

There are three layers in the modern stack:

1. **`nftables`** — the kernel module (the actual filter). Modern, supersedes `iptables`.
2. **`iptables`** — the older interface, still common, still works. Most distros now ship `iptables` as a translation layer over `nftables`.
3. **`ufw`** (Debian/Ubuntu) and **`firewalld`** (RHEL/Fedora) — friendly *higher-level* tools that generate the underlying rules. **This is what you'll actually use.**

For LFCS, know `ufw` and `firewalld` at the user level. Recognise `iptables`/`nftables` syntax in older docs.

### `ufw` — Uncomplicated Firewall (Debian/Ubuntu)

The simplest firewall manager. Default-deny incoming, default-allow outgoing.

```bash
sudo ufw status                              # current state and rules
sudo ufw enable                              # turn it on (also enables at boot)
sudo ufw disable                             # turn it off

# Allow specific services
sudo ufw allow ssh                           # allow SSH (port 22)
sudo ufw allow 80/tcp                        # by port + protocol
sudo ufw allow from 192.168.1.0/24           # from a specific network
sudo ufw allow from 10.0.0.5 to any port 5432  # specific source, specific port

# Deny / delete
sudo ufw deny 23/tcp                         # explicitly deny telnet
sudo ufw delete allow 80/tcp                 # remove a previous rule
sudo ufw status numbered                     # rules with numbers
sudo ufw delete 3                            # delete rule number 3

# Reset everything
sudo ufw reset
```

**Critical**: **before enabling ufw, allow SSH** (`sudo ufw allow ssh`). Otherwise you'll lock yourself out of a remote machine. The classic locked-out-of-cloud-VM mistake.

### `firewalld` (RHEL/Fedora) — zones-based

`firewalld` organises rules into **zones**. Each interface is assigned to a zone; the zone's rules apply to that interface's traffic. Default zones include `public`, `trusted`, `home`, `work`, `dmz`, `internal`.

```bash
sudo firewall-cmd --state                              # is it running?
sudo firewall-cmd --get-default-zone                   # current default zone
sudo firewall-cmd --list-all                           # rules for default zone
sudo firewall-cmd --get-zones                          # all zones

# Allow services
sudo firewall-cmd --add-service=ssh                    # temporary (until reboot)
sudo firewall-cmd --add-service=ssh --permanent        # permanent — survives reboot
sudo firewall-cmd --add-port=8080/tcp --permanent

# Apply permanent changes to the running config
sudo firewall-cmd --reload
```

**Two-step model**: changes are *runtime* by default (apply now, vanish on reboot). Add `--permanent` to save them; then `--reload` to apply. Or `--runtime-to-permanent` to save the current runtime state.

### `iptables` / `nftables` raw

Worth knowing at recognition level. A typical `iptables` rule chain:

```bash
sudo iptables -L -n -v                       # list rules
sudo iptables -A INPUT -p tcp --dport 22 -j ACCEPT   # allow SSH
sudo iptables -A INPUT -j DROP                       # drop everything else
```

`nftables` equivalent (newer syntax):

```bash
sudo nft list ruleset                        # show all rules
```

For LFCS, **practice with ufw** if you're on Ubuntu, **firewalld** if you're on RHEL/Fedora. You shouldn't need to write raw iptables for the exam.

## Network troubleshooting — ping, traceroute, ss

When the network "isn't working", a layered approach finds the problem fast.

### Layer 1: am I connected at all?

```bash
ip -br link                                  # all interfaces — anything DOWN?
ip -br addr                                  # do I have an IP address?
```

If your interface is `DOWN` or has no IP, fix that first — DHCP failure, cable unplugged, NetworkManager issue.

### Layer 2: can I reach the gateway?

```bash
ip route get 1.1.1.1                         # what gateway will be used?
ping -c 4 <gateway-IP>                       # is the gateway reachable?
```

**`ping`** sends ICMP echo requests. If it returns "Destination Host Unreachable" or all packets are lost, the local network has a problem (cable, switch, wrong subnet, no gateway).

### Layer 3: can I reach the wider internet?

```bash
ping -c 4 1.1.1.1                            # Cloudflare's DNS — a stable IP for testing
ping -c 4 8.8.8.8                            # Google's DNS — same idea
```

If gateway pings work but external IPs don't, the gateway / firewall / ISP is blocking outbound traffic.

### Layer 4: is DNS working?

```bash
dig +short example.com                       # do I get an answer?
dig @8.8.8.8 +short example.com              # bypass my DNS — does the wider internet work?
```

If `dig @8.8.8.8` works but `dig` alone doesn't, your DNS resolver is broken. Fix `/etc/resolv.conf` or your NetworkManager config.

### Layer 5: traceroute — where does the connection die?

```bash
traceroute example.com                       # ICMP/UDP trace
mtr example.com                              # interactive: traceroute + ping continuously
```

`traceroute` shows each hop a packet takes to reach the destination, with the latency to each. The hop that suddenly starts showing `* * *` or huge latency is your culprit (often your ISP's edge, or the destination's network).

`mtr` is `traceroute + ping` in one screen — invaluable for diagnosing intermittent network issues.

### Layer 6: is the remote port actually listening?

```bash
# Test if a TCP port is open
nc -zv example.com 443                       # netcat: "can I open a TCP connection?"
curl -v https://example.com                  # verbose curl: full request/response

# On the local machine, what ports are MY services listening on?
ss -tunlp                                    # all TCP/UDP listening, with process names (modern replacement for netstat)
ss -tn state established                     # active TCP connections
```

**`ss` is the modern `netstat`.** Use it; netstat is deprecated. Flags:

| Flag | Effect |
|---|---|
| **`-t`** | TCP |
| **`-u`** | UDP |
| **`-n`** | numeric (don't resolve names) |
| **`-l`** | listening only |
| **`-p`** | show process |
| **`-a`** | all (default skips listeners) |

A useful combination: `ss -tunlp` = "show me every TCP and UDP socket that's listening, with port numbers and which process owns it."

### Layer 7: see the actual packets — `tcpdump`

When everything else has failed, look at the wire:

```bash
sudo tcpdump -i eth0                              # everything on eth0 (overwhelming)
sudo tcpdump -i eth0 host 8.8.8.8                 # only traffic to/from 8.8.8.8
sudo tcpdump -i eth0 port 22                      # only SSH traffic
sudo tcpdump -i eth0 -n 'port 53'                 # DNS queries
sudo tcpdump -i eth0 -w capture.pcap              # save to file (open in Wireshark)
```

`tcpdump` shows you exactly what packets are flowing — perfect for "why isn't this connection completing?" investigations.

### A diagnostic flowchart

When a remote service isn't reachable:

1. **`ping <ip>`** — is the host alive at the network layer?
2. **`nc -zv <ip> <port>`** — is the specific port reachable?
3. **`dig <hostname>`** — does the name resolve?
4. **`traceroute <ip>`** — where does it fail along the path?
5. **On the server: `ss -tunlp`** — is the service actually listening?
6. **`tcpdump`** if it's still mysterious.

For LFCS, know **`ping`, `traceroute`, `ss -tunlp`, `dig`, `nc`, `tcpdump` basics**.

In [ ]:
!ping -c 2 1.1.1.1 2>/dev/null
!ss -tunlp 2>/dev/null | head -10
!dig +short example.com 2>/dev/null

## What you've learned

A compact recap to keep next to your prompt:

- **TCP/IP basics** — IP address + port = endpoint. TCP for reliable streams (SSH, HTTP), UDP for fast/lossy (DNS, video). CIDR notation: `/24` = 256-address LAN. Private ranges: `10.0.0.0/8`, `172.16-31`, `192.168`.
- **Interfaces** — `lo`, `eth0`, `enp0s3`, `wlan0`. Inspect with **`ip addr`**, **`ip link`**, **`ip -br addr`**. Modify (temporarily) with `ip addr add`/`ip link set up`.
- **`ip` replaces `ifconfig`/`route`/`netstat`** — use it in new work.
- **DHCP vs static** — most clients use DHCP, most servers use static.
- **NetworkManager + `nmcli`** — the modern config tool. `nmcli connection add type ethernet ... ipv4.method manual ipv4.addresses ...`.
- **Routing** — **`ip route`** shows table, `default via` is the gateway, `ip route get <ip>` to debug which route an IP will use.
- **DNS** — checked in order: `/etc/hosts`, then resolver. `/etc/resolv.conf` (often managed by `systemd-resolved`). **`dig +short example.com`** for queries. **`dig @8.8.8.8`** to bypass your resolver.
- **SSH** — port 22. `ssh user@host`. `~/.ssh/known_hosts` for host fingerprints. **`ssh -i KEY`** for a specific key, **`-p N`** for custom port.
- **SSH keys** — **`ssh-keygen -t ed25519`**, **`ssh-copy-id user@host`** to install. Critical perms: `~/.ssh` 700, `authorized_keys` and private keys 600.
- **`~/.ssh/config`** — `Host alias` blocks with `HostName`, `User`, `Port`, `IdentityFile`. `ProxyJump` for bastions. `Host *` for defaults.
- **`ssh-agent`** — holds decrypted keys. `ssh-add -l`, `ssh-add KEY`, `ssh-add -D`. **`ForwardAgent yes`** or `-A` for agent forwarding (use sparingly; prefer `ProxyJump`).
- **`sshd_config`** — `PermitRootLogin no`, `PasswordAuthentication no`, `AllowUsers`, `MaxAuthTries`. **`sshd -t`** to validate, **`systemctl reload sshd`** to apply. Keep one session open while testing!
- **`scp`** — `scp src user@host:dst` (note: capital `-P` for port).
- **`rsync`** — `rsync -avzh --delete src/ user@host:dst/`. **The trailing-slash rule**: `src/` copies contents, `src` copies the directory itself.
- **Firewalls** — `ufw` (Debian/Ubuntu), `firewalld` (RHEL/Fedora). Both ride on top of kernel `nftables`/`iptables`. **`sudo ufw allow ssh` BEFORE `ufw enable`** to avoid lockouts. `firewalld` is two-step: `--permanent` then `--reload`.
- **Troubleshooting layered flow** — `ping` (host alive?) → `dig` (DNS?) → `traceroute` / `mtr` (path?) → `nc -zv host port` (port open?) → `ss -tunlp` (server listening?) → `tcpdump` (what's on the wire?).
- **`ss` replaces `netstat`** — use **`ss -tunlp`** as your default.

**Practise before moving on.** Set up SSH key authentication to your practice VM end-to-end: generate a key, `ssh-copy-id`, log in without a password. Configure your firewall to allow SSH and HTTPS only. Use `nmcli` to inspect your current connection. Run `ss -tunlp` and identify each listening service. `tcpdump -i any port 53` while you run `dig` in another window — watch your DNS query go out and come back. These muscle-memory exercises are exactly what LFCS tests.

**Next:** notebook `09-services-boot-systemd-and-packages` covers what *runs* on your machine — the boot sequence, systemd units, `systemctl`, log inspection with `journalctl`, and package management with `apt`/`dnf`.